# 06 — ML: treinamento e avaliação do KNN

Código espelhado de `RETREINO/04_ml.py`. `tbl_ml` foi substituída por `desloc.gold.transacoes_ml`; `distancia` por `distancia_lalo`. O modelo, as métricas MAE/RMSE, os gráficos de erro e as 100 divisões repetidas seguem a sequência do professor.

### Treinamento

In [ ]:
from sklearn.model_selection import train_test_split

df = spark.table("desloc.gold.transacoes_ml").toPandas()
X = df.drop(columns=["vlr_pago"])
Y = df["vlr_pago"]

X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor

# Desabilitar o MLFLOW automatico do Databricks
import mlflow
mlflow.autolog(disable=True)

colunas_minmax = [
    'dia', 'dia_semana', 'mes', 'hora', 'distancia_lalo','latitude_origem',
    'longitude_origem', 'latitude_destino', 'longitude_destino'
]

preprocessador = ColumnTransformer(
    transformers=[
        ('drop', 'drop', ['periodo']),
        ('minmax', MinMaxScaler(), colunas_minmax)
    ],
    remainder='passthrough'  # mantém as outras colunas sem alteração
)

pipeline = Pipeline(steps=[
    ('preprocessamento', preprocessador),
    ('knn', KNeighborsRegressor(n_neighbors=5))
])

pipeline.fit(X_train, y_train)

In [ ]:
y_hat_train = pipeline.predict(X_train)
y_hat_test = pipeline.predict(X_test)

### Métricas

In [ ]:
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error

metrics = dict(
    mae_train = mean_absolute_error(y_train, y_hat_train),
    mae_test = mean_absolute_error(y_test, y_hat_test),
    rmse_train = root_mean_squared_error(y_train, y_hat_train),
    rmse_test = root_mean_squared_error(y_test, y_hat_test)
)

pd.DataFrame(metrics, index=[0])

In [ ]:
e_train = y_train - y_hat_train
e_test = y_test - y_hat_test

df_plot = pd.DataFrame({
    'e_train': e_train,
    'e_test': e_test,
    'y_train': y_train,
    'y_test': y_test,
})

df_plot[["e_train", "e_test"]].hist(bins=20, figsize=(10, 5));

In [ ]:
df_plot[["e_train", "y_train"]].plot(
    kind="scatter", x="y_train", y="e_train",
    grid=True, title="Train X Erro"
);

In [ ]:
df_plot[["e_test", "y_test"]].plot(
    kind="scatter", x="y_test", y="e_test",
    grid=True, title="Train X Erro"
);

In [ ]:
def print_metrics(y_train, y_test, y_hat_train, y_hat_test):
    metrics = dict(
        mae_train = mean_absolute_error(y_train, y_hat_train),
        mae_test = mean_absolute_error(y_test, y_hat_test),
        rmse_train = root_mean_squared_error(y_train, y_hat_train),
        rmse_test = root_mean_squared_error(y_test, y_hat_test)
    )

    return pd.DataFrame(metrics, index=[0])

In [ ]:
dfs = []
for i in range(0, 100):
    X_train, X_test, y_train, y_test = train_test_split(
        X, Y,
        test_size=0.2,
    )

    pipeline.fit(X_train, y_train)

    y_hat_train = pipeline.predict(X_train)
    y_hat_test = pipeline.predict(X_test)
    dfs.append(print_metrics(y_train, y_test, y_hat_train, y_hat_test))

In [ ]:
pd.concat(dfs).describe()

### Testando o Algoritmo

Interprete os valores produzidos nesta execução; os números do material do professor não são resultados desta base. Após o laço de 100 divisões, `y_test` e `y_hat_test` correspondem à última divisão, enquanto `metrics` guarda a primeira divisão.


In [ ]:
m = pd.DataFrame(metrics, index=[0])
m.columns = ["Erro em Treino", "Erro em Teste", "Erro Penalizado em Treino", "Erro Penalizado em Teste"]
m

In [ ]:
print(f"Valor total real das corridas no teste: R$ {y_test.sum():.2f}")
print(f"Valor total previsto pelo algoritmo no teste: R$ {y_hat_test.sum():.2f}")

## Explicabilidade (XAI) e limite de negócio
Cabeçalho mantido para uma etapa futura, sem alterar o código da aula. As métricas aqui não comparam o KNN com o modelo atualmente em produção. As features de calendário vêm da entrega e precisam ser substituídas por informações disponíveis antes da cotação.